# 28 - Handle Decompositions & Discrete Morse Theory

Surgery theory has an algebraic side (chain complexes, $L$-groups) and a **geometric side** (manifolds with handles). A **handle decomposition** is a way of building a manifold by successively attaching standard pieces: 0-handles (balls), 1-handles (thickened arcs), 2-handles (disks), etc. The data of which handle to attach and how it is framed *is* the surgery datum.

**Discrete Morse theory** (Forman) gives a systematic way to simplify handle decompositions: a Morse matching on the Hasse diagram of a CW complex collapses pairs of cells that cancel each other. The remaining unmatched cells are the "critical" handles.

## Learning Goals
- Understand k-handles as geometric realizations of chain complex generators
- Convert a CW complex to a handle decomposition with `cw_complex_to_handle_decomposition`
- Read `Handle` and `HandleDecomposition` Pydantic contracts
- Attach a framed handle manually and observe the change in homology
- Extract the intersection form from the 2-handles of a 4-manifold

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | CW complex = union of cells $e_k$ | `CWComplex` |
| **Algebraic** | Handle = generator of $C_k$ with attaching map and framing | `Handle`, `HandleDecomposition` |
| **Result** | Intersection form from 2-handles | `HandleDecomposition.intersection_form()` |

## Formal Background

A **$k$-handle** $H^k = D^k \times D^{n-k}$ is attached to $\partial M^{n-1}$ along $S^{k-1} \times D^{n-k} \subset \partial M$. The **attaching sphere** $S^{k-1} \times \{0\}$ is the key datum. For surgery on a 4-manifold, the 2-handles are attached along framed knots in $S^3$, and the framing determines the intersection form:
$$\lambda(h_i, h_j) = \text{linking number of attaching spheres}.$$

In [ ]:
import numpy as np
from pysurgery.core.handle_decompositions import (
    Handle, HandleDecomposition,
    cw_complex_to_handle_decomposition,
)
from pysurgery.core.complexes import CWComplex, SimplicialComplex
from pysurgery.core.intersection_forms import IntersectionForm

print("=" * 70)
print("28 - Handle Decompositions & Discrete Morse Theory: Setup Complete")
print("=" * 70)

## Part 1: What Is a Handle?

A $k$-handle is modelled on $D^k \times D^{n-k}$. The essential data:
- **Index**: $k$ (the handle is a "$k$-handle")
- **Attaching sphere**: $S^{k-1} \hookrightarrow \partial M_{k-1}$ (where to attach)
- **Framing**: trivialisation of the normal bundle of $S^{k-1}$ in $\partial M_{k-1}$
- **Core disk**: $D^k \times \{0\}$ (the generating cell)

The `Handle` Pydantic model stores these as:

In [ ]:
# Inspect the Handle model
h = Handle(
    index=1,
    dimension=4,
    attaching_sphere_class=np.array([1]),   # attaching class in H₀(∂M)
    framing_matrix=None,                    # 1-handles need no framing in dim 4
)
print("Handle fields:")
print(f"  index:                  {h.index}")
print(f"  dimension:              {h.dimension}")
print(f"  attaching_sphere_class: {h.attaching_sphere_class}")
print(f"  framing_matrix:         {h.framing_matrix}")
print()

# A 2-handle with non-trivial framing (surgery coefficient +2)
h2 = Handle(
    index=2,
    dimension=4,
    attaching_sphere_class=np.array([1, 0]),  # attaches along first generator of H₁
    framing_matrix=np.array([[2]]),           # +2 framing
)
print(f"2-handle framing_matrix: {h2.framing_matrix}")

## Part 2: CW Complex to Handle Decomposition

`cw_complex_to_handle_decomposition` applies discrete Morse theory to a CW complex:
1. Constructs the Hasse diagram of the cell poset
2. Finds a maximum acyclic matching (greedy by default, exact if `exact=True`)
3. Unmatched cells become handles; matched pairs cancel
4. Returns a `HandleDecomposition` with minimal handle count

The **theorem_tag** records which Morse theorem was applied.

In [ ]:
# Standard 4-manifolds
spaces = {
    "S⁴":  CWComplex.sphere(4),
    "CP²": CWComplex.complex_projective_space(2),
    "T⁴":  CWComplex.torus(4),
}

for name, M in spaces.items():
    hd: HandleDecomposition = cw_complex_to_handle_decomposition(M)
    by_index = {}
    for h in hd.handles:
        by_index.setdefault(h.index, 0)
        by_index[h.index] += 1
    print(f"{name}: {len(hd.handles)} handles total — {dict(sorted(by_index.items()))}")
    print(f"  exact={hd.exact}, theorem_tag={hd.theorem_tag}")

# Expected:
# S⁴:  one 0-handle + one 4-handle
# CP²: one 0-handle + one 2-handle + one 4-handle
# T⁴:  one handle per cell of T⁴ (no cancellations unless Morse matching fires)

## Part 3: Discrete Morse Matching

The discrete Morse matching cancels pairs $(\sigma^{(k)}, \tau^{(k+1)})$ when $\sigma$ is the unique face of $\tau$ (or vice versa). Each cancellation reduces the handle count by 2 without changing the homotopy type.

The greedy algorithm (default) finds a large matching fast. The exact algorithm finds the *minimum* number of critical cells but is slower for large complexes.

In [ ]:
# Compare handle counts: full CW vs Morse-reduced
T2 = SimplicialComplex.torus()
T2_cw = CWComplex.from_simplicial(T2)

hd_unreduced = cw_complex_to_handle_decomposition(T2_cw)
print("T² handle decomposition:")
print(f"  handles: {len(hd_unreduced.handles)}")
for h in hd_unreduced.handles:
    print(f"    index={h.index}, exact={h.framing_matrix is not None}")

# Betti number lower bound on handle count
print()
print(f"Betti numbers of T²: β₀={T2.betti_number(0)}, β₁={T2.betti_number(1)}, β₂={T2.betti_number(2)}")
print(f"Minimum handles = Σβₖ = {sum(T2.betti_number(k) for k in range(3))}")
print(f"Actual handles  = {len(hd_unreduced.handles)} (equality = Morse perfect)")

# Critical cells are exactly the handles
critical = [h for h in hd_unreduced.handles]
print()
print(f"Critical cells: {[h.index for h in critical]}")

## Part 4: Framed Handle Attachment

A **framed 2-handle** attachment to $S^3$ is the key surgery operation in 4-manifold theory. The framing coefficient $n \in \mathbb{Z}$ specifies how the attaching sphere $S^1 \hookrightarrow S^3$ wraps around itself. The result is a 4-manifold $W$ with $\partial W = $ the result of Dehn surgery on the knot.

The intersection form of $W$ is determined by the framing matrix of the 2-handles.

In [ ]:
# Start with S⁴ and attach a 2-handle with framing n
S4 = CWComplex.sphere(4)
hd_S4 = cw_complex_to_handle_decomposition(S4)

# Attach a 2-handle with self-framing n=+1 (gives CP² after capping)
framing_plus1 = np.array([[1]])
new_handle = Handle(
    index=2,
    dimension=4,
    attaching_sphere_class=np.array([1]),
    framing_matrix=framing_plus1,
)
hd_augmented = hd_S4.attach_handle(new_handle)
print("S⁴ + 1 framed 2-handle:")
print(f"  handles: {[(h.index, h.framing_matrix.tolist() if h.framing_matrix is not None else None) for h in hd_augmented.handles]}")

# Compare framings
for n, label in [(-2, "E₈-type"), (0, "trivial"), (+1, "CP²-type"), (+2, "exotic")]:
    hd_n = hd_S4.attach_handle(Handle(
        index=2, dimension=4,
        attaching_sphere_class=np.array([1]),
        framing_matrix=np.array([[n]])
    ))
    iform = hd_n.intersection_form()
    print(f"  framing {n:+d}: signature={iform.signature()}, unimodular={iform.is_unimodular}")

## Part 5: Intersection Form from Handle Decomposition

For a compact oriented 4-manifold $W$ with $\partial W = \emptyset$ (or $\partial W = S^3$), the **intersection form** $\lambda: H_2(W) \times H_2(W) \to \mathbb{Z}$ is computed from the framing matrices of the 2-handles.

The matrix $\lambda_{ij} = $ framing of $h_i$ if $i = j$ (self-intersection), and $= $ linking number of $h_i, h_j$ if $i \neq j$. The signature $\sigma(W)$ is the most important invariant: $\sigma(CP^2) = +1$, $\sigma(-CP^2) = -1$, $\sigma(K3) = -16$.

In [ ]:
# K3 surface: signature -16, even form E₈ ⊕ E₈ ⊕ H ⊕ H ⊕ H
# Build handle decomposition of K3 (22 handles: 1 zero + 20 two-handles + 1 four)
try:
    K3 = CWComplex.K3_surface()
    hd_K3 = cw_complex_to_handle_decomposition(K3)
    iform_K3 = hd_K3.intersection_form()
    print("K3 surface intersection form:")
    print(f"  rank:         {iform_K3.rank}")
    print(f"  signature:    {iform_K3.signature()}")   # should be -16
    print(f"  even:         {iform_K3.is_even}")        # should be True (E₈ is even)
    print(f"  unimodular:   {iform_K3.is_unimodular}")
    print(f"  exact:        {hd_K3.exact}")
except Exception as e:
    print(f"K3 not available as built-in: {e}")
    print("Constructing E₈ plumbing instead:")
    # E₈ plumbing: 8 two-handles with E₈ framing matrix
    E8_matrix = np.array([
        [-2, 1, 0, 0, 0, 0, 0, 0],
        [ 1,-2, 1, 0, 0, 0, 0, 0],
        [ 0, 1,-2, 1, 0, 0, 0, 1],
        [ 0, 0, 1,-2, 1, 0, 0, 0],
        [ 0, 0, 0, 1,-2, 1, 0, 0],
        [ 0, 0, 0, 0, 1,-2, 1, 0],
        [ 0, 0, 0, 0, 0, 1,-2, 0],
        [ 0, 0, 1, 0, 0, 0, 0,-2],
    ])
    iform_E8 = IntersectionForm(matrix=E8_matrix)
    print(f"  E₈ signature:  {iform_E8.signature()}")   # -8
    print(f"  E₈ even:       {iform_E8.is_even}")
    print(f"  E₈ unimodular: {iform_E8.is_unimodular}")

## Part 6: Connection to Algebraic Surgery (NB 12)

Handle decompositions provide the **geometric realization** of algebraic surgery moves:
- **Elementary surgery**: cancel a pair of handles (algebraic: exact couple splitting)
- **Handle slide**: change framing by adding a column (algebraic: elementary row/column op)
- **Stabilisation**: add a cancelling pair $H^k \oplus H^{n-k}$ (algebraic: add hyperbolic summand to form)

After surgery, the boundary $\partial W$ changes by a Dehn surgery on the attaching knots. The algebraic counterpart is the Wall surgery obstruction in $L_n(\mathbb{Z}[\pi])$.

In [ ]:
# Demonstrate handle cancellation: add a cancelling pair to S⁴
# A 0-handle + 1-handle cancels (algebraically: exact split of chain complex)
S4_hd = cw_complex_to_handle_decomposition(CWComplex.sphere(4))
print(f"S⁴ handle count: {len(S4_hd.handles)}")

# Stabilise by adding H (one +1 and one -1 handle pair)
stabilised = S4_hd.stabilize(framing=+1)
print(f"Stabilised count: {len(stabilised.handles)}")
print(f"New intersection form signature: {stabilised.intersection_form().signature()}")
print("(signature shifts by +1 per +1 stabilisation)")

print()
print("Key identity:")
print("  Topological surgery ↔ Algebraic surgery")
print("  Handle attachment  ↔ Generator addition to chain complex")
print("  Framing change     ↔ Elementary column operation on boundary matrix")
print("  Cancellation pair  ↔ Acyclic exact couple splitting")

## Summary Checklist

- [x] Understood k-handles as geometric realizations of chain complex generators
- [x] Converted CW complexes to handle decompositions via `cw_complex_to_handle_decomposition`
- [x] Read `Handle.index`, `framing_matrix`, `attaching_sphere_class`
- [x] Understood discrete Morse matching as handle cancellation
- [x] Attached framed 2-handles and observed their effect on the intersection form
- [x] Extracted the intersection form from a 4-manifold handle decomposition
- [x] Connected handle moves to algebraic surgery operations

## Next Steps
- **NB 07**: Intersection forms — can now be derived directly from handle data
- **NB 29**: Rational surgery uses the intersection form from handle decompositions
- **NB 34**: The capstone pipeline uses handle decompositions at step 5